# 03 — Preprocessing

This notebook implements preprocessing requirements motivated by `02_eda.ipynb`. It creates no models, predictions, or evaluation metrics. Every fitted preprocessing object uses training data only; the verified tree-ready and ANN-ready splits are saved under `data/processed/`.

## 1. Load Data and Add Structural Features

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MaxAbsScaler, OneHotEncoder, PowerTransformer, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

project_root = Path.cwd()
if not (project_root / "data" / "raw").exists():
    project_root = project_root.parent
data_path = project_root / "data" / "raw" / "wind_turbine_detection.csv"
df = pd.read_csv(data_path)
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="raise")
df = df.sort_values(["turbine_id", "timestamp"]).reset_index(names="source_index")
print(f"Loaded rows: {len(df):,}; source features plus target/identifiers: {df.shape[1] - 1}")
print(f"Timestamp range: {df['timestamp'].min()} to {df['timestamp'].max()}")

### Preprocessing feature scope

No engineered predictors are created in this notebook. `hour` and `day_of_week`, used only for an EDA heatmap in notebook 02, are excluded because this is snapshot classification and calendar-pattern dependence is not established. Wind bins and a below-cut-in flag also remain EDA concepts; continuous `wind_speed_mps` is retained. The existing `rated_power_kW` field is treated as categorical and one-hot encoded so neither model branch is forced to assume an ordinal distance between turbine classes.

In [ ]:
below_cutin_mask = df["wind_speed_mps"] < 3
structural_zero_audit = pd.DataFrame({
    "feature": ["power_output_kW", "rotor_speed_rpm", "generator_speed_rpm"],
    "below_cutin_rows": int(below_cutin_mask.sum()),
    "zero_count_below_cutin": [int(df.loc[below_cutin_mask, c].eq(0).sum()) for c in ["power_output_kW", "rotor_speed_rpm", "generator_speed_rpm"]],
})
structural_zero_audit["zero_pct_below_cutin"] = structural_zero_audit["zero_count_below_cutin"] / structural_zero_audit["below_cutin_rows"] * 100
display(structural_zero_audit)
print("Engineered model features created:", [])
print("EDA-only fields excluded: ['hour', 'day_of_week', 'wind_speed_bin', 'below_cutin']")
print("Observed rated_power_kW categories:", sorted(df["rated_power_kW"].unique().tolist()))

## 2. Grouped Outer Test Split and Temporal Validation Split

A stratified grouped outer split holds out entire turbines for final testing, preventing turbine identity from crossing into test data. Within the remaining development turbines, the earliest 80% of each turbine forms training data and the latest 20% forms validation data. A temporal boundary moves forward when necessary so a contiguous 10-minute failure episode is not divided. This nested design tests unseen-turbine generalization while retaining a later validation period for threshold and model selection.

In [ ]:
outer_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
development_idx, test_idx = next(outer_splitter.split(df, y=df["failure"], groups=df["turbine_id"]))
development = df.iloc[development_idx].copy()
test = df.iloc[test_idx].copy()

def move_boundary_past_episode(group, boundary):
    boundary = min(max(int(boundary), 1), len(group) - 1)
    while boundary < len(group) and group.iloc[boundary - 1]["failure"] == 1 and group.iloc[boundary]["failure"] == 1:
        boundary += 1
    return boundary

train_parts, validation_parts = [], []
for turbine_id, group in development.groupby("turbine_id", sort=True):
    group = group.sort_values("timestamp").reset_index(drop=True)
    boundary = move_boundary_past_episode(group, np.floor(len(group) * 0.80))
    train_parts.append(group.iloc[:boundary])
    validation_parts.append(group.iloc[boundary:])
train = pd.concat(train_parts, ignore_index=True)
validation = pd.concat(validation_parts, ignore_index=True)
splits = {"train": train, "validation": validation, "test": test}

split_report = pd.DataFrame({
    name: {"rows": len(part), "turbines": part["turbine_id"].nunique(),
           "failure_count": int(part["failure"].sum()), "failure_pct": part["failure"].mean() * 100,
           "start": part["timestamp"].min(), "end": part["timestamp"].max()}
    for name, part in splits.items()
}).T
display(split_report)
for name, part in splits.items():
    print(f"{name} turbine IDs: {sorted(part['turbine_id'].unique().tolist())}")
print("Train/test turbine overlap:", sorted(set(train["turbine_id"]) & set(test["turbine_id"])))
print("Validation/test turbine overlap:", sorted(set(validation["turbine_id"]) & set(test["turbine_id"])))

In [ ]:
boundary_episode_splits = 0
for turbine_id in train["turbine_id"].unique():
    train_last = train.loc[train["turbine_id"] == turbine_id, "timestamp"].max()
    validation_first = validation.loc[validation["turbine_id"] == turbine_id, "timestamp"].min()
    combined = development.loc[development["turbine_id"] == turbine_id].sort_values("timestamp")
    left = combined.loc[combined["timestamp"] == train_last, "failure"].iloc[0]
    right = combined.loc[combined["timestamp"] == validation_first, "failure"].iloc[0]
    boundary_episode_splits += int(left == 1 and right == 1)
print(f"Development rows retained: {len(train) + len(validation) == len(development)}")
print(f"Test rows retained: {len(test) == len(test_idx)}")
print(f"Failure episodes split at train/validation boundaries: {boundary_episode_splits}")

## 3. Turbine-ID Target-Encoding Audit

Grouped cross-fitting is used to test turbine-ID target encoding without allowing a turbine's labels to encode its own training rows. Because each grouped fold holds out complete turbine IDs, those IDs are unseen by the fold encoder and receive the fold's global prior. The completely held-out test turbines likewise receive the training prior. If this produces no stable turbine-specific training signal, turbine ID is excluded rather than using leaky naive target encoding.

In [ ]:
group_encoder = GroupKFold(n_splits=5)
train_turbine_encoding = pd.Series(index=train.index, dtype=float)
for encoder_train_idx, encoder_holdout_idx in group_encoder.split(train, groups=train["turbine_id"]):
    encoder_train = train.iloc[encoder_train_idx]
    encoder_holdout = train.iloc[encoder_holdout_idx]
    prior = encoder_train["failure"].mean()
    mapping = encoder_train.groupby("turbine_id")["failure"].mean()
    train_turbine_encoding.iloc[encoder_holdout_idx] = encoder_holdout["turbine_id"].map(mapping).fillna(prior)
full_train_prior = train["failure"].mean()
full_train_mapping = train.groupby("turbine_id")["failure"].mean()
validation_turbine_encoding = validation["turbine_id"].map(full_train_mapping).fillna(full_train_prior)
test_turbine_encoding = test["turbine_id"].map(full_train_mapping).fillna(full_train_prior)
encoding_report = pd.Series({
    "grouped_oof_training_unique_values": train_turbine_encoding.nunique(),
    "grouped_oof_training_std": train_turbine_encoding.std(),
    "validation_unique_values": validation_turbine_encoding.nunique(),
    "heldout_test_unique_values": test_turbine_encoding.nunique(),
    "heldout_test_rows_using_global_prior": int(test_turbine_encoding.eq(full_train_prior).sum()),
    "training_global_prior": full_train_prior,
})
display(encoding_report.to_frame("value"))
use_turbine_target_encoding = False
print("Turbine target encoding retained as model feature:", use_turbine_target_encoding)
print("Reason: grouped holdout provides no identity-specific encoding for unseen turbines; naive encoding is prohibited.")

## 4. Feature Policy and Outlier Preservation

The six provenance-risk history fields identified earlier are conservatively excluded. IQR outliers in the five failure-enriched condition signals are retained without clipping, winsorization, or row deletion. No outlier indicator or sensor-range feature is created.

In [ ]:
leakage_candidates_excluded = ["prior_fault_count", "hours_since_last_maintenance", "component_age_days",
                               "operating_hours_total", "cumulative_energy_MWh", "load_cycles"]
preserved_outlier_features = ["generator_winding_temp_C", "gearbox_bearing_temp_C",
                              "drivetrain_vibration_rms_mmps", "vib_fft_bearing_bpfo", "oil_particle_count"]
outlier_audit = []
for column in preserved_outlier_features:
    q1, q3 = train[column].quantile([0.25, 0.75])
    iqr = q3 - q1
    flag = train[column].notna() & ((train[column] < q1 - 1.5 * iqr) | (train[column] > q3 + 1.5 * iqr))
    outlier_audit.append({"feature": column, "training_iqr_outliers_retained": int(flag.sum()),
                          "failure_share_pct": train.loc[flag, "failure"].mean() * 100})
display(pd.DataFrame(outlier_audit))
print("Rows removed for outliers:", 0)
print("Outlier or sensor-range features created:", [])
print("Leakage candidates excluded:", leakage_candidates_excluded)

## 5. Training-Only Multicollinearity Reduction

Correlation is recalculated on training data. Generator speed, gearbox-oil temperature, and generator-bearing temperature are removed in favor of rotor speed, gearbox-bearing temperature, and retained component-specific temperatures. The four FFT frequency bands remain available to tree models; the ANN/regularization-sensitive representation compresses their transformed values with train-fitted PCA.

In [ ]:
administrative = ["source_index", "timestamp", "turbine_id", "failure"]
eligible_numeric = [c for c in train.select_dtypes(include="number").columns if c not in administrative + leakage_candidates_excluded]
train_corr = train[eligible_numeric].corr()
upper = train_corr.where(np.triu(np.ones(train_corr.shape), k=1).astype(bool))
before_pairs = (upper.stack().rename("correlation").reset_index()
                .rename(columns={"level_0": "feature_1", "level_1": "feature_2"}))
before_pairs["absolute_correlation"] = before_pairs["correlation"].abs()
before_high = before_pairs.loc[before_pairs["absolute_correlation"] >= 0.90].sort_values("absolute_correlation", ascending=False)
display(before_high)
redundancy_drops = ["generator_speed_rpm", "gearbox_oil_temp_C", "generator_bearing_temp_C"]
numeric_features = [c for c in eligible_numeric if c not in redundancy_drops and c != "rated_power_kW"]
retained_corr = train[numeric_features].corr()
retained_upper = retained_corr.where(np.triu(np.ones(retained_corr.shape), k=1).astype(bool))
after_pairs = int((retained_upper.abs().stack() >= 0.90).sum())
print(f"Training pairs with |r| >= 0.90 before pruning: {len(before_high)}")
print(f"Correlation-redundant features dropped: {redundancy_drops}")
print(f"Training pairs with |r| >= 0.90 after pruning: {after_pairs}")

## 6. Train-Fitted Imputation, Skew Correction, Encoding, PCA, and Scaling

Median imputation addresses only missing values and leaves observed zeros intact. The EDA-specified skewed features receive Yeo–Johnson transformation, which supports negative and zero values. For power output, `standardize=False` is followed by `MaxAbsScaler`, preserving exact zeros. Rotor speed also uses `MaxAbsScaler`. Other ANN inputs use train-fitted standard scaling. The existing five-level `rated_power_kW` field is one-hot encoded so no ordinal spacing is imposed, which is robust for both tree and ANN branches.

In [ ]:
fft_features = ["vib_fft_bearing_bpfo", "vib_fft_bearing_bpfi", "vib_fft_gearmesh", "vib_fft_sideband"]
skewed_non_fft = ["turbulence_intensity", "generator_winding_temp_C", "drivetrain_vibration_rms_mmps",
                  "tower_vibration_mmps", "oil_particle_count", "oil_pressure_bar", "blade_pitch_angle_deg"]
zero_preserving_skewed = ["power_output_kW"]
zero_preserving_regular = ["rotor_speed_rpm"]
categorical_features = ["rated_power_kW"]
rated_power_levels = sorted(train["rated_power_kW"].dropna().unique().tolist())
used_special = set(fft_features + skewed_non_fft + zero_preserving_skewed + zero_preserving_regular)
regular_numeric = [c for c in numeric_features if c not in used_special]

tree_preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), numeric_features),
    ("categorical", OneHotEncoder(categories=[rated_power_levels], handle_unknown="ignore", sparse_output=False), categorical_features),
], remainder="drop", verbose_feature_names_out=False)

ann_preprocessor = ColumnTransformer([
    ("fft_pca", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("yeo_johnson", PowerTransformer(method="yeo-johnson", standardize=False)),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=3, svd_solver="full")),
    ]), fft_features),
    ("skewed", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("yeo_johnson", PowerTransformer(method="yeo-johnson", standardize=False)),
        ("scaler", StandardScaler()),
    ]), skewed_non_fft),
    ("power_zero_preserving", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("yeo_johnson", PowerTransformer(method="yeo-johnson", standardize=False)),
        ("scaler", MaxAbsScaler()),
    ]), zero_preserving_skewed),
    ("rotor_zero_preserving", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", MaxAbsScaler()),
    ]), zero_preserving_regular),
    ("regular", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), regular_numeric),
    ("categorical", OneHotEncoder(categories=[rated_power_levels], handle_unknown="ignore", sparse_output=False), categorical_features),
], remainder="drop", verbose_feature_names_out=False)

model_input_columns = numeric_features + categorical_features
X = {name: part[model_input_columns].copy() for name, part in splits.items()}
y = {name: part["failure"].astype("int8").copy() for name, part in splits.items()}
X_tree = {"train": tree_preprocessor.fit_transform(X["train"])}
X_ann = {"train": ann_preprocessor.fit_transform(X["train"])}
for name in ["validation", "test"]:
    X_tree[name] = tree_preprocessor.transform(X[name])
    X_ann[name] = ann_preprocessor.transform(X[name])
print("Tree output features:", tree_preprocessor.get_feature_names_out().tolist())
print("ANN output features:", ann_preprocessor.get_feature_names_out().tolist())
pca = ann_preprocessor.named_transformers_["fft_pca"].named_steps["pca"]
print(f"FFT PCA components retained: {pca.n_components_} of {len(fft_features)}")
print(f"FFT PCA cumulative explained variance: {pca.explained_variance_ratio_.sum():.6f}")

In [ ]:
eda_skew_features = skewed_non_fft + fft_features + zero_preserving_skewed
skew_rows = []
for column in eda_skew_features:
    values = train[[column]]
    imputed = SimpleImputer(strategy="median").fit_transform(values)
    transformed = PowerTransformer(method="yeo-johnson", standardize=False).fit_transform(imputed).ravel()
    skew_rows.append({"feature": column, "training_skew_before": pd.Series(imputed.ravel()).skew(),
                      "training_skew_after_yeo_johnson": pd.Series(transformed).skew()})
skew_diagnostics = pd.DataFrame(skew_rows)
display(skew_diagnostics)
print(f"Maximum absolute post-transform skew: {skew_diagnostics['training_skew_after_yeo_johnson'].abs().max():.6f}")
power_pipeline = ann_preprocessor.named_transformers_["power_zero_preserving"]
rotor_pipeline = ann_preprocessor.named_transformers_["rotor_zero_preserving"]
power_zero_transformed = power_pipeline.transform(pd.DataFrame({"power_output_kW": [0.0]}))[0, 0]
rotor_zero_transformed = rotor_pipeline.transform(pd.DataFrame({"rotor_speed_rpm": [0.0]}))[0, 0]
print(f"Transformed power zero: {power_zero_transformed:.6f}")
print(f"Transformed rotor-speed zero: {rotor_zero_transformed:.6f}")

## 7. Class-Imbalance Policy

Split balance is reported rather than altered with synthetic samples. Balanced class weights are calculated from training labels for later model fitting. SMOTE is intentionally not used because synthetic interpolation could dilute the failure-enriched temperature and vibration extremes. PR-AUC and failure recall remain later evaluation criteria, not preprocessing outputs.

In [ ]:
balance_report = pd.DataFrame({
    name: {"normal_count": int((labels == 0).sum()), "failure_count": int((labels == 1).sum()),
           "failure_pct": labels.mean() * 100}
    for name, labels in y.items()
}).T
display(balance_report)
classes = np.array([0, 1])
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y["train"].to_numpy())
class_weight = dict(zip(classes.tolist(), weights.tolist()))
print("Training-derived balanced class weights for later modeling:", class_weight)
print("Synthetic oversampling applied:", False)

## 8. Pipeline Validation

In [ ]:
validation_rows = []
for name in splits:
    validation_rows.append({
        "split": name, "source_rows": len(splits[name]),
        "tree_rows": X_tree[name].shape[0], "tree_features": X_tree[name].shape[1],
        "ann_rows": X_ann[name].shape[0], "ann_features": X_ann[name].shape[1],
        "tree_nonfinite": int((~np.isfinite(X_tree[name])).sum()),
        "ann_nonfinite": int((~np.isfinite(X_ann[name])).sum()),
        "target_preserved": bool(np.array_equal(y[name].to_numpy(), splits[name]["failure"].to_numpy())),
    })
validation_report = pd.DataFrame(validation_rows).set_index("split")
display(validation_report)
all_checks_passed = bool(
    (validation_report[["tree_nonfinite", "ann_nonfinite"]] == 0).all().all()
    and validation_report["target_preserved"].all()
    and boundary_episode_splits == 0
    and not (set(train["turbine_id"]) & set(test["turbine_id"]))
)
print("All preprocessing validation checks passed:", all_checks_passed)

## 9. Save and Reload Model-Ready Splits

The six CSV files contain transformed features plus the unchanged `failure` target. Identifiers and timestamps are deliberately excluded from model inputs. Tree and ANN representations are saved separately because their transformations and feature dimensions differ.

In [ ]:
if not all_checks_passed:
    raise RuntimeError("Preprocessing validation failed; processed files will not be written.")

processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
tree_feature_names = tree_preprocessor.get_feature_names_out().tolist()
ann_feature_names = ann_preprocessor.get_feature_names_out().tolist()
saved_paths = {}
for split_name in splits:
    tree_frame = pd.DataFrame(X_tree[split_name], columns=tree_feature_names)
    tree_frame["failure"] = y[split_name].to_numpy()
    ann_frame = pd.DataFrame(X_ann[split_name], columns=ann_feature_names)
    ann_frame["failure"] = y[split_name].to_numpy()

    tree_path = processed_dir / f"03_{split_name}_tree.csv"
    ann_path = processed_dir / f"03_{split_name}_ann.csv"
    tree_frame.to_csv(tree_path, index=False)
    ann_frame.to_csv(ann_path, index=False)
    saved_paths[f"{split_name}_tree"] = tree_path
    saved_paths[f"{split_name}_ann"] = ann_path

save_report_rows = []
for artifact_name, artifact_path in saved_paths.items():
    reloaded = pd.read_csv(artifact_path)
    split_name, representation = artifact_name.rsplit("_", 1)
    expected_rows = len(splits[split_name])
    expected_features = X_tree[split_name].shape[1] if representation == "tree" else X_ann[split_name].shape[1]
    save_report_rows.append({
        "artifact": artifact_name,
        "path": str(artifact_path.relative_to(project_root)),
        "shape": reloaded.shape,
        "file_size_mib": artifact_path.stat().st_size / 1024**2,
        "failure_count": int(reloaded["failure"].sum()),
        "failure_rate_pct": reloaded["failure"].mean() * 100,
        "rows_verified": len(reloaded) == expected_rows,
        "columns_verified": reloaded.shape[1] == expected_features + 1,
        "target_verified": np.array_equal(reloaded["failure"].to_numpy(), y[split_name].to_numpy()),
    })
save_report = pd.DataFrame(save_report_rows).set_index("artifact")
display(save_report)
saved_outputs_verified = bool(save_report[["rows_verified", "columns_verified", "target_verified"]].all().all())
print("Saved files verified after reload:", saved_outputs_verified)
print(f"Processed files written: {len(saved_paths)}")
# Export the exact split handoff and fitted ANN representation together.
import hashlib
import joblib
for split_name, part in splits.items():
    part[["source_index", "timestamp", "turbine_id", "failure"]].to_csv(
        processed_dir / f"03_{split_name}_metadata.csv", index=False)
np.savez_compressed(processed_dir / "03_ann_arrays.npz",
    **{f"X_{name}": X_ann[name].astype("float32") for name in splits},
    **{f"y_{name}": y[name].to_numpy() for name in splits})
joblib.dump(ann_preprocessor, processed_dir / "03_ann_preprocessor.joblib")
artifact_names = ["03_ann_arrays.npz", "03_ann_preprocessor.joblib"] + [
    f"03_{name}_metadata.csv" for name in splits]
manifest = {name: hashlib.sha256((processed_dir / name).read_bytes()).hexdigest()
            for name in artifact_names}
manifest["raw_sha256"] = hashlib.sha256(data_path.read_bytes()).hexdigest()
(processed_dir / "03_ann_manifest.json").write_text(__import__("json").dumps(manifest, indent=2))
print("Saved verified split metadata, ANN arrays, fitted preprocessor and integrity manifest.")


## 10. Exploratory Lead-Time Feature Policy (Not Implemented)

No global slope or rolling lead-time feature is created in this preprocessing baseline. Step 02 found inconsistent directions around the six longest episodes, so any future lead-time features must be evaluated separately by turbine or episode type, computed using past-only windows, and fitted entirely within training folds.

## Key Findings Recap

- No engineered predictors were created: EDA-only `hour`, `day_of_week`, `wind_speed_bin`, and `below_cutin` fields were excluded, while the five observed `rated_power_kW` categories were one-hot encoded without imposing ordinal distances.
- The below-cut-in audit covered 14,875 rows; zero-preserving preprocessing maps observed zero power and rotor speed to exactly 0.000000 after transformation without adding a below-cut-in feature.
- The nested split produced 84,336 training rows across 12 turbines, 21,072 later validation rows from those turbines, and 26,352 test rows from three entirely held-out turbines, with no test-turbine overlap or split failure episodes.
- Grouped cross-fitted turbine encoding produced one global-prior value for all 26,352 held-out test rows, so turbine target encoding was not retained and naive target encoding was not used.
- No IQR outliers were removed or winsorized, no outlier or sensor-range indicator features were created, and all six leakage-candidate history fields were excluded.
- Training data contained 16 pairs with `|r| >= 0.90`; dropping `generator_speed_rpm`, `gearbox_oil_temp_C`, and `generator_bearing_temp_C` reduced the retained count to 7.
- Yeo–Johnson correction reduced the maximum absolute skew among the 12 specified features to 0.331399; the ANN FFT block retained three PCA components explaining 0.780693 of variance.
- Training-derived balanced class weights were 0.511381 for normal rows and 22.465637 for failure rows; no synthetic oversampling was applied, and six model-ready split files passed reload, shape, and target-preservation checks.